# Sentiment Classification Project - GloVe Baseline

The goal is to build a clean English-GloVe baseline, measure where it has vocabulary coverage, and use the validation results to motivate later improvements - especially regarding mutlilingual datasets.

In [1]:
from pathlib import Path
import re
import zipfile
import urllib.request

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

# Load data

In [2]:
train_full = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

print(train_full.shape)
print(test_df.shape)
print(train_full.head())
print(train_full["label"].value_counts().sort_index())

(252000, 3)
(168000, 2)
   id                                           sentence  label
0   0  Moderner Weihnachtsbaum in weiß\n\nDieses Jahr...      4
1   1  Passt wie angegossen\n\nTasche kam schnell und...      4
2   2  Schlechte Qualität\n\nIch habe sehr lange auf ...      1
3   3  Bestellung nie angekommen\n\n-5 Sterne..... Am...      0
4   4  Für mich gar nicht gut\n\nMacht das Makeup gar...      1
label
0    50400
1    50400
2    50400
3    50400
4    50400
Name: count, dtype: int64


# Clean data

Preprocessing is kept intentionally light. GloVe lookup does benefit from lowercasing and whitespace normalization, but aggressive preprocessing could remove useful sentiment cues such as punctuation, negation, and emojis.

In [3]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_full["text_clean"] = train_full["sentence"].apply(clean_text)
test_df["text_clean"] = test_df["sentence"].apply(clean_text)

train_full[["sentence", "text_clean", "label"]].head()

,sentence,text_clean,label
0,Moderner Weihnachtsbaum in weiß\n\nDieses Jahr...,Moderner Weihnachtsbaum in weiß Dieses Jahr wo...,4
1,Passt wie angegossen\n\nTasche kam schnell und...,Passt wie angegossen Tasche kam schnell und gu...,4
2,Schlechte Qualität\n\nIch habe sehr lange auf ...,Schlechte Qualität Ich habe sehr lange auf die...,1
3,Bestellung nie angekommen\n\n-5 Sterne..... Am...,Bestellung nie angekommen -5 Sterne..... Am 15...,0
4,Für mich gar nicht gut\n\nMacht das Makeup gar...,Für mich gar nicht gut Macht das Makeup gar ni...,1


# Simple language detection
## ToDo: Find better language classification method

This is a heuristic, not a production language detector. It is useful for analysis because the dataset is mostly German and English. We count common German/English function words and give German a small boost when umlauts or `ß` occur.


In [4]:
DE_WORDS = set("""
der die das und ist nicht ich ein eine einer einem einen zu mit für auf den im in es auch sehr aber von
sie er dem des dass habe hat war sind so wie man nur noch wenn kein keine keinen als doch nach kann zum
zur bei oder um aus
""".split())

EN_WORDS = set("""
the and is not i a an to with for on it also very but of in you this that have has was are so as only
if no my we they he she from or by at be been can do does did would could should after when what which
""".split())

WORD_RE = re.compile(r"[A-Za-zÄÖÜäöüß]+(?:'[A-Za-z]+)?")

def tokenize_words(text):
    return [token.lower() for token in WORD_RE.findall(str(text))]

def detect_language_simple(text):
    tokens = tokenize_words(text)
    de_score = sum(token in DE_WORDS for token in tokens)
    en_score = sum(token in EN_WORDS for token in tokens)
    de_score += sum(any(char in token for char in "äöüß") for token in tokens)
    
    if de_score >= en_score + 2:
        return "de"
    if en_score >= de_score + 2:
        return "en"
    return "uncertain"

train_full["lang"] = train_full["text_clean"].apply(detect_language_simple)
test_df["lang"] = test_df["text_clean"].apply(detect_language_simple)

print(train_full["lang"].value_counts(normalize=True))
pd.crosstab(train_full["label"], train_full["lang"], normalize="index").round(3)

lang
de           0.481504
en           0.478111
uncertain    0.040385
Name: proportion, dtype: float64


lang,de,en,uncertain
label,,,
0,0.487,0.480,0.032
1,0.492,0.484,0.024
2,0.487,0.485,0.027
3,0.477,0.478,0.045
4,0.464,0.463,0.073


# Build Validation Set
We use 90% of the reviews for training, and the remaining 10% for validation

In [5]:
train_df, val_df = train_test_split(
    train_full,
    test_size=0.1,
    stratify=train_full["label"],
    random_state=RANDOM_STATE,
)

Y_train = train_df["label"].to_numpy()
Y_val = val_df["label"].to_numpy()

print(train_df.shape, val_df.shape)

(226800, 5) (25200, 5)


# Load pretrained GloVe

Download `glove.6B.zip` from Stanford and extract `glove.6B.100d.txt` into `embeddings/`, or let the cell below download and extract it.

English GloVe is expected to have much weaker coverage on German reviews. We keep that limitation visible because it is one of the main things we want to analyze.

In [6]:
GLOVE_DIR = Path("embeddings")
GLOVE_ZIP = GLOVE_DIR / "glove.6B.zip"
GLOVE_PATH = GLOVE_DIR / "glove.6B.100d.txt"
GLOVE_URL = "https://nlp.stanford.edu/data/glove.6B.zip"

GLOVE_DIR.mkdir(exist_ok=True)

if not GLOVE_PATH.exists():
    if not GLOVE_ZIP.exists():
        print("Downloading GloVe. This file is large, so it may take a while.")
        urllib.request.urlretrieve(GLOVE_URL, GLOVE_ZIP)
    with zipfile.ZipFile(GLOVE_ZIP) as zf:
        zf.extract(GLOVE_PATH.name, path=GLOVE_DIR)

print(GLOVE_PATH)

embeddings/glove.6B.100d.txt


In [7]:
def load_glove(path):
    embeddings = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            word = parts[0]
            vector = np.asarray(parts[1:], dtype=np.float32)
            embeddings[word] = vector
    dim = len(next(iter(embeddings.values())))
    return embeddings, dim

glove, EMBEDDING_DIM = load_glove(GLOVE_PATH)
print(len(glove), EMBEDDING_DIM)

400000 100


# Measure vocabulary coverage

Coverage tells us how much of our review text can actually use a pretrained vector. Low coverage means the model is averaging over fewer informative words, which is especially important for German reviews when using English GloVe.

In [8]:
def coverage_report(texts, embeddings):
    total_tokens = 0
    covered_tokens = 0
    vocab = set()
    covered_vocab = set()
    
    for text in texts:
        tokens = tokenize_words(text)
        total_tokens += len(tokens)
        for token in tokens:
            vocab.add(token)
            if token in embeddings:
                covered_tokens += 1
                covered_vocab.add(token)
    
    return pd.Series({
        "token_coverage": covered_tokens / max(total_tokens, 1),
        "vocab_coverage": len(covered_vocab) / max(len(vocab), 1),
        "tokens": total_tokens,
        "vocab": len(vocab),
    })

coverage_by_language = train_df.groupby("lang")["text_clean"].apply(
    lambda texts: coverage_report(texts, glove)
).unstack()

coverage_overall = coverage_report(train_df["text_clean"], glove)

print("Overall coverage")
display(coverage_overall)
display(coverage_by_language)

Overall coverage


token_coverage    8.451420e-01
vocab_coverage    3.262246e-01
tokens            8.437293e+06
vocab             1.154450e+05
dtype: float64

,token_coverage,vocab_coverage,tokens,vocab
lang,,,,
de,0.695051,0.153827,4041845.0,86214.0
en,0.985661,0.865036,4327148.0,35891.0
uncertain,0.824671,0.624735,68300.0,8498.0


# Mean GloVe features

Each review becomes the average of all GloVe vectors found in the text. Unknown words are skipped. If no known word exists, we return a zero vector.

In [9]:
def mean_glove_vector(text, embeddings, dim):
    vectors = [embeddings[token] for token in tokenize_words(text) if token in embeddings]
    if not vectors:
        return np.zeros(dim, dtype=np.float32)
    return np.mean(vectors, axis=0)

def build_mean_glove_matrix(texts, embeddings, dim):
    return np.vstack([mean_glove_vector(text, embeddings, dim) for text in texts])

X_train_mean = build_mean_glove_matrix(train_df["text_clean"], glove, EMBEDDING_DIM)
X_val_mean = build_mean_glove_matrix(val_df["text_clean"], glove, EMBEDDING_DIM)

print(X_train_mean.shape, X_val_mean.shape)

(226800, 100) (25200, 100)


# Train Logistic Regression on mean GloVe

Now we train a logistic regression classifier...

In [10]:
scaler_mean = StandardScaler()
X_train_mean_scaled = scaler_mean.fit_transform(X_train_mean)
X_val_mean_scaled = scaler_mean.transform(X_val_mean)

mean_model = LogisticRegression(C=1.0, max_iter=1000, n_jobs=-1)
mean_model.fit(X_train_mean_scaled, Y_train)

Y_train_pred_mean = mean_model.predict(X_train_mean_scaled)
Y_val_pred_mean = mean_model.predict(X_val_mean_scaled)

/cluster/courses/cil/envs/envs/text-classification/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


# TF-IDF weighted GloVe features

Plain averaging treats all words equally. TF-IDF weighting gives more influence to words that are informative in a review and less influence to very common words.

In [11]:
tfidf = TfidfVectorizer(tokenizer=tokenize_words, preprocessor=None, lowercase=False, min_df=2)
tfidf.fit(train_df["text_clean"])

idf = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))
default_idf = max(tfidf.idf_)

def tfidf_weighted_glove_vector(text, embeddings, dim, idf_lookup, default_weight):
    weighted_vectors = []
    weights = []
    for token in tokenize_words(text):
        if token in embeddings:
            weight = idf_lookup.get(token, default_weight)
            weighted_vectors.append(embeddings[token] * weight)
            weights.append(weight)
    if not weighted_vectors:
        return np.zeros(dim, dtype=np.float32)
    return np.sum(weighted_vectors, axis=0) / np.sum(weights)

def build_tfidf_glove_matrix(texts, embeddings, dim, idf_lookup, default_weight):
    return np.vstack([
        tfidf_weighted_glove_vector(text, embeddings, dim, idf_lookup, default_weight)
        for text in texts
    ])

X_train_tfidf = build_tfidf_glove_matrix(train_df["text_clean"], glove, EMBEDDING_DIM, idf, default_idf)
X_val_tfidf = build_tfidf_glove_matrix(val_df["text_clean"], glove, EMBEDDING_DIM, idf, default_idf)

print(X_train_tfidf.shape, X_val_tfidf.shape)

/cluster/courses/cil/envs/envs/text-classification/lib/python3.14/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


(226800, 100) (25200, 100)


In [12]:
scaler_tfidf = StandardScaler()
X_train_tfidf_scaled = scaler_tfidf.fit_transform(X_train_tfidf)
X_val_tfidf_scaled = scaler_tfidf.transform(X_val_tfidf)

tfidf_model = LogisticRegression(C=1.0, max_iter=1000, n_jobs=-1)
tfidf_model.fit(X_train_tfidf_scaled, Y_train)

Y_train_pred_tfidf = tfidf_model.predict(X_train_tfidf_scaled)
Y_val_pred_tfidf = tfidf_model.predict(X_val_tfidf_scaled)

/cluster/courses/cil/envs/envs/text-classification/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


# GloVe + handcrafted text features

These features capture signals that averaged embeddings ignore, such as review length, punctuation intensity, uppercase emphasis, and rough negation counts.

In [13]:
NEGATION_WORDS = set("not no never n't nicht kein keine keinen niemals ohne".split())

def handcrafted_features(text):
    text = str(text)
    tokens = tokenize_words(text)
    token_count = len(tokens)
    char_count = len(text)
    exclamation_count = text.count("!")
    question_count = text.count("?")
    uppercase_words = sum(token.isupper() and len(token) > 1 for token in re.findall(r"\b\w+\b", text))
    negation_count = sum(token in NEGATION_WORDS for token in tokens)
    known_tokens = sum(token in glove for token in tokens)
    coverage = known_tokens / max(token_count, 1)
    
    return np.array([
        np.log1p(char_count),
        np.log1p(token_count),
        exclamation_count,
        question_count,
        uppercase_words,
        negation_count,
        coverage,
    ], dtype=np.float32)

def build_handcrafted_matrix(texts):
    return np.vstack([handcrafted_features(text) for text in texts])

X_train_hand = build_handcrafted_matrix(train_df["text_clean"])
X_val_hand = build_handcrafted_matrix(val_df["text_clean"])

X_train_combined = np.hstack([X_train_tfidf, X_train_hand])
X_val_combined = np.hstack([X_val_tfidf, X_val_hand])

print(X_train_combined.shape, X_val_combined.shape)

(226800, 107) (25200, 107)


In [14]:
scaler_combined = StandardScaler()
X_train_combined_scaled = scaler_combined.fit_transform(X_train_combined)
X_val_combined_scaled = scaler_combined.transform(X_val_combined)

combined_model = LogisticRegression(C=1.0, max_iter=1000, n_jobs=-1)
combined_model.fit(X_train_combined_scaled, Y_train)

Y_train_pred_combined = combined_model.predict(X_train_combined_scaled)
Y_val_pred_combined = combined_model.predict(X_val_combined_scaled)

/cluster/courses/cil/envs/envs/text-classification/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


# Evaluate models

In [15]:
def competition_score(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    return  1.0 - (mae / 4.0)

def evaluate_predictions(name, y_train, y_train_pred, y_val, y_val_pred):
    return pd.Series({
        "model": name,
        "train_score": competition_score(y_train, y_train_pred),
        "val_score": competition_score(y_val, y_val_pred),
        "val_mae": mean_absolute_error(y_val, y_val_pred),
        "val_accuracy": np.mean(y_val == y_val_pred),
        "val_macro_f1": f1_score(y_val, y_val_pred, average="macro"),
    })

results = pd.DataFrame([
    evaluate_predictions("Mean GloVe + Logistic Regression", Y_train, Y_train_pred_mean, Y_val, Y_val_pred_mean),
    evaluate_predictions("TF-IDF GloVe + Logistic Regression", Y_train, Y_train_pred_tfidf, Y_val, Y_val_pred_tfidf),
    evaluate_predictions("TF-IDF GloVe + Handcrafted + Logistic Regression", Y_train, Y_train_pred_combined, Y_val, Y_val_pred_combined),
])

results.sort_values("val_score", ascending=False)

,model,train_score,val_score,val_mae,val_accuracy,val_macro_f1
2,TF-IDF GloVe + Handcrafted + Logistic Regression,0.770554,0.769435,0.922262,0.411349,0.406024
0,Mean GloVe + Logistic Regression,0.769188,0.768710,0.925159,0.413651,0.406665
1,TF-IDF GloVe + Logistic Regression,0.752200,0.752778,0.988889,0.393651,0.385315


In [16]:
val_analysis = val_df.copy()
val_analysis["pred"] = Y_val_pred_combined
val_analysis["abs_error"] = (val_analysis["label"] - val_analysis["pred"]).abs()

language_scores = []
for lang, group in val_analysis.groupby("lang"):
    language_scores.append(pd.Series({
        "lang": lang,
        "n": len(group),
        "score": competition_score(group["label"], group["pred"]),
        "mae": mean_absolute_error(group["label"], group["pred"]),
        "accuracy": np.mean(group["label"] == group["pred"]),
    }))

pd.DataFrame(language_scores).sort_values("score", ascending=False)

,lang,n,score,mae,accuracy
2,uncertain,1010,0.796040,0.815842,0.505941
1,en,11938,0.780365,0.878539,0.421427
0,de,12252,0.756591,0.973637,0.393732


In [17]:
conf_matrix = confusion_matrix(Y_val,Y_val_pred_combined, labels=[0,1,2,3,4])
pd.DataFrame(conf_matrix, index=[0,1,2,3,4], columns=[0,1,2,3,4])

,0,1,2,3,4
0,2761,1064,549,333,333
1,1312,1594,1082,683,369
2,830,1220,1412,1085,493
3,435,574,923,1645,1463
4,350,277,401,1058,2954


# Error analysis

In [18]:
very_wrong = val_analysis.sort_values("abs_error", ascending=False).head(10)

for _, row in very_wrong.iterrows():
    print("-" * 80)
    print(f"Language: {row['lang']} | Actual: {row['label']} | Predicted: {row['pred']} | Error: {row['abs_error']}")
    print(row["text_clean"][:700])

--------------------------------------------------------------------------------
Language: en | Actual: 0 | Predicted: 4 | Error: 4
This is a long wait!! Hope it’s I. Good condition!! Still waiting for this cd of movie!! Why taking so long??
--------------------------------------------------------------------------------
Language: en | Actual: 0 | Predicted: 4 | Error: 4
Sad kid...john deere We didn't receive it. We received a frame instead.
--------------------------------------------------------------------------------
Language: en | Actual: 0 | Predicted: 4 | Error: 4
wont charge will not charge have tried my charger and my brothers
--------------------------------------------------------------------------------
Language: en | Actual: 0 | Predicted: 4 | Error: 4
You get what you pay for! SUPER thin/see through.
--------------------------------------------------------------------------------
Language: de | Actual: 4 | Predicted: 0 | Error: 4
Grabkerze Kerzen brennen komplett runter, 

# Make test submission

We use the strongest validation model from this notebook. At this stage that is usually the combined feature model, but this should be checked after running the notebook.

In [19]:
X_test_tfidf = build_tfidf_glove_matrix(test_df["text_clean"], glove, EMBEDDING_DIM, idf, default_idf)
X_test_hand = build_handcrafted_matrix(test_df["text_clean"])
X_test_combined = np.hstack([X_test_tfidf, X_test_hand])
X_test_combined_scaled = scaler_combined.transform(X_test_combined)

submit_preds = combined_model.predict(X_test_combined_scaled)

submission = pd.DataFrame({
    "id": test_df["id"],
    "label": submit_preds,
})

Path("submissions").mkdir(exist_ok=True)
submission_path = "submissions/glove_logreg_submission.csv"
submission.to_csv(submission_path, index=False)

print(submission_path)
submission.head()

submissions/glove_logreg_submission.csv


,id,label
0,0,0
1,1,2
2,2,3
3,3,2
4,4,3
